# Импорты и загрузка данных

In [1]:
from pathlib import Path
import sys
import time
import json
import re
from difflib import SequenceMatcher

from transformers import AutoTokenizer

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from io_utils import load_jsonl
from prompts import build_messages

In [2]:
dev_data = load_jsonl("../data/dev.jsonl")

len(dev_data)

40

In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

# Подсчет токенов

In [4]:
def count_input_tokens(product: dict) -> int:
    """Вычисляет количество токенов полного входа."""
    messages = build_messages(product)

    return len(
        tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
        )
    )

In [5]:
def count_product_tokens_without_reviews(product: dict) -> int:
    """Вычисляет количество токенов карточки без отзывов."""
    product_without_reviews = product.copy()
    product_without_reviews["reviews"] = []

    return count_input_tokens(product_without_reviews)

# Нормализация текста и удаление дубликатов

In [6]:
def normalize_review(review: str) -> str:
    """Нормализует текст отзыва для сравнения."""
    review = review.lower()
    review = re.sub(r"[^\w\s]", " ", review)
    review = re.sub(r"\s+", " ", review)
    return review.strip()

In [7]:
def remove_exact_duplicates(reviews: list[str]) -> list[str]:
    """Удаляет точные дубликаты, сохраняя порядок отзывов."""
    unique_reviews = []
    seen = set()

    for review in reviews:
        normalized_review = normalize_review(review)

        if normalized_review not in seen:
            seen.add(normalized_review)
            unique_reviews.append(review)

    return unique_reviews

In [8]:
def remove_similar_reviews(
    reviews: list[str],
    similarity_threshold: float = 0.9,
) -> list[str]:
    """Удаляет вложенные и сильно похожие отзывы."""
    reviews = sorted(reviews, key=len, reverse=True)
    unique_reviews = []

    for review in reviews:
        normalized_review = normalize_review(review)

        for saved_review in unique_reviews:
            normalized_saved = normalize_review(saved_review)

            if normalized_review in normalized_saved:
                break

            similarity = SequenceMatcher(
                None,
                normalized_review,
                normalized_saved,
            ).ratio()

            if similarity >= similarity_threshold:
                break
        else:
            unique_reviews.append(review)

    return unique_reviews


# Стратегии сокращения длинного входа

In [9]:
def fit_reviews_to_token_limit(
        product: dict,
        reviews: list[str],
        max_input_tokens: int = 1500
) -> list[str]:
    """Удаляет последние отзывы, пока полный вход не уложится в лимит."""
    reviews = reviews.copy()

    while reviews:
        shortened_product = product.copy()
        shortened_product["reviews"] = reviews

        if count_input_tokens(shortened_product) <= max_input_tokens:
            break

        reviews.pop()

    return reviews

In [10]:
def selection_of_reviews(review_budget: int, product: dict) -> list[str]:
    """Поочередно отбирает длинные и короткие отзывы."""
    reviews = sorted(product["reviews"], key=len)

    selected = []
    left = 0
    right = len(reviews) - 1

    while left <= right:
        review = reviews[right]
        num_tokens = len(tokenizer.encode(review))

        if num_tokens <= review_budget:
            selected.append(review)
            review_budget -= num_tokens

        right -= 1

        if left > right:
            break

        review = reviews[left]
        num_tokens = len(tokenizer.encode(review))

        if num_tokens <= review_budget:
            selected.append(review)
            review_budget -= num_tokens

        left += 1

    return fit_reviews_to_token_limit(product, selected)

In [11]:
def truncation_of_reviews(review_budget: int, product: dict) -> list[str]:
    """Отбирает отзывы по порядку до исчерпания бюджета."""
    selected = []

    for review in product["reviews"]:
        num_tokens = len(tokenizer.encode(review))

        if num_tokens <= review_budget:
            selected.append(review)
            review_budget -= num_tokens
        else:
            break

    return fit_reviews_to_token_limit(product, selected)

# Сравнение стратегий

In [12]:
def run_strategy(name: str, select_reviews):
    """Обрабатывает длинный вход и сохраняет результаты стратегии."""
    rows = []

    for product in dev_data:
        started = time.perf_counter()

        original_reviews = product["reviews"]
        original_tokens = count_input_tokens(product)

        exact_reviews = original_reviews
        similar_reviews = original_reviews
        selected_reviews = original_reviews

        used_exact_dedup = False
        used_similar_dedup = False
        used_selection = False

        if original_tokens > 1500:
            used_exact_dedup = True
            exact_reviews = remove_exact_duplicates(original_reviews)

            prepared_product = product.copy()
            prepared_product["reviews"] = exact_reviews

            if count_input_tokens(prepared_product) > 1500:
                used_similar_dedup = True
                similar_reviews = remove_similar_reviews(exact_reviews)

                prepared_product["reviews"] = similar_reviews

                if count_input_tokens(prepared_product) > 1500:
                    used_selection = True

                    review_budget = (
                        1500
                        - count_product_tokens_without_reviews(prepared_product)
                    )

                    selected_reviews = select_reviews(
                        review_budget,
                        prepared_product,
                    )
                else:
                    selected_reviews = similar_reviews
            else:
                similar_reviews = exact_reviews
                selected_reviews = exact_reviews

        shortened_product = product.copy()
        shortened_product["reviews"] = selected_reviews
        input_tokens = count_input_tokens(shortened_product)

        rows.append({
            "product_id": product["product_id"],
            "reviews": selected_reviews,
            "reviews_before": len(original_reviews),
            "reviews_after_exact": len(exact_reviews),
            "reviews_after_similar": len(similar_reviews),
            "selected_reviews": len(selected_reviews),
            "original_tokens": original_tokens,
            "input_tokens": input_tokens,
            "used_exact_dedup": used_exact_dedup,
            "used_similar_dedup": used_similar_dedup,
            "used_selection": used_selection,
            "time_ms": (time.perf_counter() - started) * 1000,
        })

    output_dir = (
        PROJECT_ROOT
        / "outputs"
        / "experiments"
        / "long_input_strategy"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    with (output_dir / f"{name}.jsonl").open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    {
                        "product_id": row["product_id"],
                        "reviews": row["reviews"],
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )

    return rows

In [13]:
def calculate_metrics(name: str, rows: list[dict]) -> dict:
    """Вычисляет метрики обработки длинного входа."""
    long_rows = [row for row in rows if row["original_tokens"] > 1500]

    exact_removed = sum(
        row["reviews_before"] - row["reviews_after_exact"]
        for row in long_rows
    )

    similar_removed = sum(
        row["reviews_after_exact"] - row["reviews_after_similar"]
        for row in long_rows
    )

    return {
        "strategy": name,
        "cards_total": len(rows),
        "cards_over_1500": len(long_rows),
        "exact_duplicates_removed": exact_removed,
        "cards_after_exact_over_1500": sum(
            row["used_similar_dedup"] for row in rows
        ),
        "similar_reviews_removed": similar_removed,
        "cards_after_similar_over_1500": sum(
            row["used_selection"] for row in rows
        ),
        "avg_selected_reviews": (
            sum(row["selected_reviews"] for row in long_rows) / len(long_rows)
            if long_rows else 0
        ),
        "avg_input_tokens": sum(
            row["input_tokens"] for row in rows
        ) / len(rows),
        "max_input_tokens": max(row["input_tokens"] for row in rows),
        "avg_time_ms": sum(
            row["time_ms"] for row in rows
        ) / len(rows),
    }

In [14]:
long_short_rows = run_strategy(
    "long_short_reviews",
    selection_of_reviews,
)

truncation_rows = run_strategy(
    "truncation",
    truncation_of_reviews,
)

results = [
    calculate_metrics("long_short_reviews", long_short_rows),
    calculate_metrics("truncation", truncation_rows),
]

results_path = (
    PROJECT_ROOT
    / "outputs"
    / "experiments"
    / "long_input_strategy"
    / "results.txt"
)

with results_path.open("w", encoding="utf-8") as f:
    for result in results:
        line = (
            f"{result['strategy']}: "
            f"long cards {result['cards_over_1500']}/{result['cards_total']}; "
            f"exact duplicates removed {result['exact_duplicates_removed']}; "
            f"after exact dedup {result['cards_after_exact_over_1500']} still long; "
            f"similar reviews removed {result['similar_reviews_removed']}; "
            f"selection required {result['cards_after_similar_over_1500']}; "
            f"selected reviews {result['avg_selected_reviews']:.1f} avg; "
            f"input {result['avg_input_tokens']:.1f} avg, "
            f"{result['max_input_tokens']} max tokens; "
            f"time {result['avg_time_ms']:.1f} ms\n"
        )

        f.write(line)
        print(line, end="")

long_short_reviews: long cards 6/40; exact duplicates removed 462; after exact dedup 6 still long; similar reviews removed 91; selection required 6; selected reviews 29.0 avg; input 1195.0 avg, 1499 max tokens; time 69.5 ms
truncation: long cards 6/40; exact duplicates removed 462; after exact dedup 6 still long; similar reviews removed 91; selection required 6; selected reviews 26.3 avg; input 1195.5 avg, 1495 max tokens; time 61.8 ms
